In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import timm
import os, math, random, logging
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.transforms import functional as TF
import torchvision.utils as vutils
import matplotlib.pyplot as plt

from accelerate import Accelerator
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from diffusers import DDIMScheduler

class Autoencoder(nn.Module):
    def __init__(self, in_dim=25, bottleneck_dim=3, hidden_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, int(hidden_dim/4)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/4)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/2)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim)), nn.ReLU(), nn.LayerNorm(int(hidden_dim)), nn.Dropout(0.1),
            nn.Linear(hidden_dim, int(hidden_dim/2)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/2)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/4)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, int(hidden_dim/4)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/4)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), int(hidden_dim/2)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/2)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), hidden_dim), nn.ReLU(), nn.LayerNorm(int(hidden_dim)), nn.Dropout(0.1),
            nn.Linear(hidden_dim, int(hidden_dim/2)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/2)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/2), int(hidden_dim/4)), nn.ReLU(), nn.LayerNorm(int(hidden_dim/4)), nn.Dropout(0.1),
            nn.Linear(int(hidden_dim/4), in_dim),
        )
    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)

# z3d 缩放常量
_MIN_VALS = torch.tensor([-69.761505, -75.65188, -77.16103], dtype=torch.float32)
_MAX_VALS = torch.tensor([ 88.969406,  65.244896, 67.13518 ], dtype=torch.float32)
_RANGE    = _MAX_VALS - _MIN_VALS
THRESH_WHITE_01 = 250.0/255.0
@torch.no_grad()
def fractions_from_batch_rgb(imgs_m11, ae_model):
    """
    imgs_m11: [B,3,H,W] ∈ [0,1]；组成统计时剔除“白像素”（RGB>250/255）
    返回: fractions [B,25]、valid_pixels [B]
    """
    device = next(ae_model.parameters()).device
    B, _, H, W = imgs_m11.shape
    x01 = imgs_m11
    mv = _MIN_VALS.to(device); rg = _RANGE.to(device); thr = THRESH_WHITE_01

    fracs = []; pixs = []
    for i in range(B):
        rgb = x01[i].to(device)                        # [3,H,W]
        flat = rgb.permute(1,2,0).reshape(-1,3)
        white = ((flat > thr)).all(dim=1)               # 只在统计时剔除白
        valid = flat[~white]
        if valid.numel() == 0:
            fracs.append(torch.zeros(25, device="cpu")); pixs.append(0); continue
        z = valid * rg + mv                            # 恢复 z3d
        logits = ae_model.decoder(z)                   # [N,25]
        pred = torch.argmax(logits, dim=1)             # 0..24
        cnt  = torch.bincount(pred, minlength=25).float().cpu()
        fracs.append(cnt / cnt.sum().clamp_min(1))
        pixs.append(int(cnt.sum().item()))
    return torch.stack(fracs, 0), np.array(pixs, dtype=np.int64)


In [ ]:
# ============================================================
# 级联：冻结小模型(64x64 latent) → 采样200步得 z_cond → 训练大模型(512x512 pixel)
# 依赖: torch, diffusers, accelerate, torchvision, PIL, numpy, tqdm, matplotlib
# ============================================================
# -----------------------------
# Utils & Modules (from your code / minimal edits)
# -----------------------------
class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0 = dim0
        self.dim1 = dim1
    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

class PositionalEncoding2D(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.register_buffer('pos_embed', self.build_sincos_encoding(num_patches, dim), persistent=False)
    def build_sincos_encoding(self, num_patches, dim):
        pe = torch.zeros(num_patches, dim)
        position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1, num_patches, dim]
    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]

class ResidualBlock2d(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim=None):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.act1  = nn.SiLU()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.act2  = nn.SiLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.skip  = nn.Conv2d(in_channels, out_channels, 1) if in_channels!=out_channels else nn.Identity()
        if time_dim is not None:
            self.to_time = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_channels))
        else:
            self.to_time = None

    def forward(self, x, t_emb=None):
        h = self.conv1(self.act1(self.norm1(x)))
        if self.to_time is not None and t_emb is not None:
            # FiLM-like add
            h = h + self.to_time(t_emb)[:, :, None, None]
        h = self.conv2(self.act2(self.norm2(h)))
        return h + self.skip(x)

# -----------------------------
# Cond Encoder (your original for stage-1 tokens)
# -----------------------------
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels)
        )
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)

class CondEncoder(nn.Module):
    def __init__(self, in_channels=4, out_channels=736, num_tokens=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64), # [B, 64, 64, 64]
            nn.AvgPool2d(2), # [B, 64, 32, 32]
            ResidualBlock(64, 128),
            nn.AvgPool2d(2), # [B, 128, 16, 16]
            ResidualBlock(128, 256),
            nn.AvgPool2d(2), # [B, 256, 8, 8]
            nn.Conv2d(256, out_channels, kernel_size=1) # [B, 736, 8, 8]
        )
        self.proj = nn.Sequential(
            nn.Flatten(2),  # [B, 736, 64]
            Transpose(-1, -2),   # [B, 64, 736]
        )
        self.pos_embed = PositionalEncoding2D(num_patches=num_tokens, dim=out_channels)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        feat = self.encoder(x)          # [B, 736, 8, 8]
        tokens = self.proj(feat)        # [B, 64, 736]
        tokens = self.pos_embed(tokens) # [B, 64, 736]
        tokens = self.norm(tokens)
        return tokens

# -----------------------------
# Dataset (unchanged except确保mask索引正确)
# -----------------------------
class OutpaintDataset(Dataset):
    def __init__(self, root_dir, img_size=512, masks_per_image=100):
        self.img_files = [os.path.join(root_dir, f) for f in os.listdir(root_dir)]
        self.img_size = img_size
        self.masks_per_image = masks_per_image
        self.transform = transforms.Compose([
            transforms.RandomChoice([
                transforms.Lambda(lambda x: x),
                transforms.Lambda(lambda x: TF.rotate(x, 90)),
                transforms.Lambda(lambda x: TF.rotate(x, 180)),
                transforms.Lambda(lambda x: TF.rotate(x, 270))
            ]),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.img_files) * self.masks_per_image

    def _gen_random_bbox(self):
        mode = random.random()
        if random.random() < 0.5:
            aspect_ratio = random.uniform(1, 33)
        else:
            aspect_ratio = random.uniform(0.03, 1)
        area = random.uniform(0.1, 0.4)**2
        w = min(np.sqrt(area * aspect_ratio), 0.99)
        h = min(np.sqrt(area / aspect_ratio), 0.99)
        x1 = random.uniform(0.05, 0.99 - w)
        y1 = random.uniform(0.05, 0.99 - h)
        return (torch.tensor(x1, dtype=torch.float16),
                torch.tensor(y1, dtype=torch.float16),
                torch.tensor(x1 + w, dtype=torch.float16),
                torch.tensor(y1 + h, dtype=torch.float16))

    def __getitem__(self, idx):
        img_idx = idx // self.masks_per_image
        img = Image.open(self.img_files[img_idx]).convert("RGB")
        img = self.transform(img)

        # bbox→mask（通道顺序 [C,H,W]）
        H, W = img.shape[1], img.shape[2]
        x1, y1, x2, y2 = self._gen_random_bbox()
        x1i, x2i = int(x1*W), int(x2*W)
        y1i, y2i = int(y1*H), int(y2*H)
        mask = torch.zeros_like(img)
        # 注意：先行(y)后列(x)
        mask[:, y1i:y2i, x1i:x2i] = 1.0

        masked_img = img * (1 - mask)
        bbox = torch.tensor([x1, y1, x2, y2], dtype=torch.float32)
        return masked_img, img, bbox

In [ ]:
class OutpaintTrainer_new:
    def __init__(self, pretrained_path="runwayml/stable-diffusion-v1-5"):
        # Initialize Accelerator for FP16 mixed precision
        self.accelerator = Accelerator(mixed_precision='fp16')
        self.loss_history = []
        self.val_loss_history = []

        # Dataset and DataLoader
        train_data = OutpaintDataset("drive/MyDrive/resize1")#, masks_per_image=60)
        val_data = OutpaintDataset("drive/MyDrive/resizeval1")#, masks_per_image=40)
        self.train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
        self.val_loader = DataLoader(val_data, batch_size=4, shuffle=False)

        # Load model components
        self.vae = AutoencoderKL.from_pretrained(pretrained_path, subfolder="vae")
        self.unet = UNet2DConditionModel.from_pretrained(pretrained_path, subfolder="unet")

        # Coordinate Encoder
        self.coord_encoder = nn.Sequential(nn.Linear(4, 32), nn.GELU())

        # Conditional Projection Layer

        self.cond_proj = CondEncoder()

        # Prepare components for mixed precision
        components = [self.vae, self.unet, self.coord_encoder, self.cond_proj, self.train_loader, self.val_loader]
        self.vae, self.unet, self.coord_encoder, self.cond_proj, self.train_loader, self.val_loader = \
            self.accelerator.prepare(*components)

        # Optimizer
        self.optimizer = torch.optim.AdamW(
            list(self.unet.parameters()) +
            list(self.coord_encoder.parameters()) +
            list(self.cond_proj.parameters()),
            lr = 2e-5)
        self.optimizer = self.accelerator.prepare(self.optimizer)

        # Noise Scheduler
        self.noise_scheduler = DDPMScheduler.from_pretrained(
            pretrained_path, subfolder="scheduler")

        # Freeze VAE
        self.vae.requires_grad_(False)
        self.accelerator.register_for_checkpointing(self.unet, self.coord_encoder, self.cond_proj, self.optimizer)

In [ ]:
# -----------------------------
# Stage-1 Sampler: 64x64 latent 采样 200步 → z_cond (冻结小模型)
# -----------------------------
class Stage1Sampler(nn.Module):
    """
    用你原来的小模型做 latent 采样：给 masked_imgs + bbox，构造 encoder_hidden_states，
    从纯噪声出发用 DDPMScheduler 反推 200 步，得到 z_cond: [B, Cz, 64,64]
    """
    def __init__(self, pretrained_path="runwayml/stable-diffusion-v1-5", stage1_checkpoint="drive/MyDrive/checkpoint_LR", device="cuda"):
        super().__init__()
        self.device = device
        trainer = OutpaintTrainer_new()
        accelerator = trainer.accelerator
        # 可选：加载你之前fine-tune的checkpoint（accelerate保存的state）
        if stage1_checkpoint is not None and os.path.exists(stage1_checkpoint):
            try:
                accelerator.load_state(stage1_checkpoint)
                _acc = accelerator
                self.vae, self.unet, self.coord_encoder, self.cond_proj = _acc.unwrap_model(trainer.vae), _acc.unwrap_model(trainer.unet), _acc.unwrap_model(trainer.coord_encoder), _acc.unwrap_model(trainer.cond_proj)
                print(f"[Stage1Sampler] Loaded checkpoint from {stage1_checkpoint}")
            except Exception as e:
                print(f"[Stage1Sampler] Warning: failed to load checkpoint: {e}")

        for m in [self.vae, self.unet, self.coord_encoder, self.cond_proj]:
            m.eval()
            for p in m.parameters():
                p.requires_grad_(False)

        # 小模型 scheduler（用diffusers现成，步数=200）
        self.scheduler = DDPMScheduler.from_pretrained(pretrained_path, subfolder="scheduler")

    def _latent_mask_from_bbox(self, bbox, H=64, W=64):
      """
      bbox: [B,4] with (x1,y1,x2,y2) in [0,1]
      return: [B,1,H,W] with 1 on masked region (to be denoised), 0 elsewhere (keep known)
      """
      B = bbox.shape[0]
      device = bbox.device
      mask = torch.zeros(B, 1, H, W, device=device, dtype=torch.float32)
      for i in range(B):
        x1, y1, x2, y2 = bbox[i]
        x1i, x2i = int(x1.item() * W), int(x2.item() * W)
        y1i, y2i = int(y1.item() * H), int(y2.item() * H)
        mask[i, :, y1i:y2i, x1i:x2i] = 1.0
      return mask

    @torch.no_grad()
    def sample_latent(self, masked_imgs, bbox, steps=200):
        """
        masked_imgs: [B,3,512,512] in [-1,1]
        bbox:        [B,4] in [0,1]
        return:      z_cond [B,4,64,64] (only masked region denoised)
        """
        device = self.device
        masked_imgs = masked_imgs.to(device)
        bbox = bbox.to(device)

        # 1) 编码已知图像 → latent（作为“已知”拷回源）
        masked_latents = self.vae.encode(masked_imgs).latent_dist.sample()
        masked_latents = masked_latents * self.vae.config.scaling_factor   # [B,4,64,64]

        # 2) 条件 tokens（来自 masked_latents + bbox）
        tokens = self.cond_proj(masked_latents)                             # [B,64,736]
        coord_feats = self.coord_encoder(bbox)                              # [B,32]
        cond = torch.cat([tokens, coord_feats.unsqueeze(1).expand(-1, 64, -1)], dim=-1)  # [B,64,768]

        # 3) latent 掩膜（1=需要去噪的“洞”，0=已知区域）
        B, Cz, H, W = masked_latents.shape
        lmask = self._latent_mask_from_bbox(bbox, H=H, W=W)                 # [B,1,H,W]
        lmask = lmask.to(masked_latents.dtype)
        # broadcast到通道
        lmask_c = lmask.expand(-1, Cz, -1, -1)                              # [B,4,H,W]

        # 4) 初始化：洞里随机噪声，已知区域用 masked_latents
        latents = torch.randn_like(masked_latents) * self.scheduler.init_noise_sigma
        latents = latents * lmask_c + masked_latents * (1.0 - lmask_c)

        # 5) 采样 200 步：每步前后都把已知区域强制拷回
        self.scheduler.set_timesteps(steps, device=device)
        for t in self.scheduler.timesteps:
           # 已知区域先钉住（数值更稳）
            latents = latents * lmask_c + masked_latents * (1.0 - lmask_c)

            noise_pred = self.unet(latents, t, encoder_hidden_states=cond).sample
            latents = self.scheduler.step(noise_pred, t, latents).prev_sample

            # 再次钉住（确保任何数值漂移不会污染已知区域）
            latents = latents * lmask_c + masked_latents * (1.0 - lmask_c)

        return latents  # [B,4,64,64]

In [ ]:
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast

# ===== 批量版：用 OutpaintDataset 做离线预计算（支持多样本并行）=====
def precompute_latents(
    root_dir,
    out_dir,
    masks_per_image=100,
    steps_stage1=200,
    stage1_pretrained="runwayml/stable-diffusion-v1-5",
    stage1_checkpoint="drive/MyDrive/checkpoint_LR",          # 传 None 则只用HF预训练
    device="cuda",
    batch_size=16,                    # <<—— 调大以吃满GPU（先从 2/4 试）
    num_workers=4,                   # CPU 解码线程数
    amp_dtype=torch.float16          # AMP 半精度推理
):
    """
    逐条从 OutpaintDataset 取 (masked_img, img, bbox)，
    批量喂入 Stage1Sampler.sample_latent 得到 z_cond，
    保存 {z_cond, bbox, masked_img, target_img} 到 out_dir/*.pt，
    并写 index.jsonl。
    """
    import os, json, torch
    from tqdm.auto import tqdm

    os.makedirs(out_dir, exist_ok=True)
    index_path = os.path.join(out_dir, "index.jsonl")

    # 数据集（保持你的随机旋转等增强）
    ds = OutpaintDataset(root_dir=root_dir, img_size=512, masks_per_image=masks_per_image)
    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,               # 顺序稳定，便于用 idx 还原 (base, k)
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False
    )

    # 小级采样器（冻结）
    sampler = Stage1Sampler(
        pretrained_path=stage1_pretrained,
        stage1_checkpoint=stage1_checkpoint if stage1_checkpoint and os.path.exists(stage1_checkpoint) else None,
        device=device
    )

    def toCPU16(t): return t.detach().cpu().half()

    # 全局样本计数器（用于算 img_idx & k）
    global_idx = 0

    with open(index_path, "w") as fw:
        pbar = tqdm(dl, desc=f"Precomputing from {root_dir}")
        for batch in pbar:
            masked_img, target_img, bbox = batch  # [B,3,512,512], [B,3,512,512], [B,4]

            # 上 GPU（非阻塞）
            masked_b = masked_img.to(device, non_blocking=True)
            bbox_b   = bbox.to(device, non_blocking=True)

            # —— 批量小级 latent inpainting 采样（200 步）——
            with torch.no_grad():
                # AMP 半精度：更快更省显存
                with autocast():
                    z_cond = sampler.sample_latent(masked_b, bbox_b, steps=steps_stage1)  # [B,4,64,64]

            B = masked_b.size(0)
            for b in range(B):
                idx = global_idx + b
                img_idx = idx // ds.masks_per_image
                k       = idx %  ds.masks_per_image
                base = os.path.splitext(os.path.basename(ds.img_files[img_idx]))[0]
                pt_path = os.path.join(out_dir, f"{base}_k{k:03d}.pt")

                torch.save({
                    "z_cond":     toCPU16(z_cond[b]),          # [4,64,64], fp16
                    "bbox":       bbox[b].cpu().tolist(),      # [4] in [0,1]
                    "masked_img": toCPU16(masked_img[b]),      # [3,512,512], fp16, 原始增强后图
                    "target_img": toCPU16(target_img[b]),      # [3,512,512], fp16
                    "src_path":   ds.img_files[img_idx],
                    "k":          int(k)
                }, pt_path)

                fw.write(json.dumps({"pt": pt_path}) + "\n")

            global_idx += B

            # 适度清理显存
            if global_idx % (batch_size * 10) == 0:
                torch.cuda.empty_cache()

    print(f"[Precompute] Done. Saved index at {index_path}")


In [ ]:
precompute_latents(
    root_dir="drive/MyDrive/resize1",
    out_dir="drive/MyDrive/precomputed_cascade/train",
    masks_per_image=60,
    steps_stage1=200,
    stage1_pretrained="runwayml/stable-diffusion-v1-5",
    stage1_checkpoint="drive/MyDrive/checkpoint_LR",
    device="cuda"
)

[Stage1Sampler] Loaded checkpoint from drive/MyDrive/checkpoint_LR


Precomputing from drive/MyDrive/resize1:   0%|          | 0/248 [00:00<?, ?it/s]

/tmp/ipython-input-1876526702.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[Precompute] Done. Saved index at drive/MyDrive/precomputed_cascade/train/index.jsonl


In [ ]:
precompute_latents(
    root_dir="drive/MyDrive/resizeval1",
    out_dir="drive/MyDrive/precomputed_cascade/val",
    masks_per_image=40, steps_stage1=200,
    stage1_pretrained="runwayml/stable-diffusion-v1-5",
    stage1_checkpoint="drive/MyDrive/checkpoint_merfish",
    device="cuda"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

[Stage1Sampler] Loaded checkpoint from drive/MyDrive/checkpoint_LR


Precomputing from drive/MyDrive/resizeval1:   0%|          | 0/10 [00:00<?, ?it/s]

/tmp/ipython-input-1876526702.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[Precompute] Done. Saved index at drive/MyDrive/precomputed_cascade/val/index.jsonl


In [ ]:
import os, torch
import torchvision.utils as vutils
from diffusers import AutoencoderKL

@torch.no_grad()
def visualize_saved_latent(pt_path, vae, out_path="latent_vis.png"):
    """
    读取之前保存的 .pt 文件，解码 z_cond 并保存图像
    """
    pack = torch.load(pt_path, map_location="cpu")
    z = pack["z_cond"].unsqueeze(0).float()   # [1,4,64,64]

    device = next(vae.parameters()).device
    z = z.to(device)

    # VAE decode 需要先除以 scaling_factor
    z = z / vae.config.scaling_factor
    img = vae.decode(z).sample              # [-1,1]
    img = (img.clamp(-1,1) + 1) / 2         # [0,1]

    vutils.save_image(img, out_path, nrow=1)
    print(f"saved visualization to {out_path}")

# ==== 用法示例 ====
vae = AutoencoderKL.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="vae").cuda().eval()

# 指定一个之前保存的 latent 文件
visualize_saved_latent("drive/MyDrive/202509vae_compressed_latent_pix_featureprompt/val/region_35_k010.pt", vae, "foo_vis.png")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: 'drive/MyDrive/202509vae_compressed_latent_pix_featureprompt/val/region_35_k010.pt'

In [ ]:
import math as _m
import torch
import torch.nn as nn
import torch.nn.functional as F

def _g(n, c):  # GroupNorm 分组自适应
    return _m.gcd(n, c) or 1

class SinCosPos2D(nn.Module):
    """ 生成 4 通道位置编码: [sin(x), cos(x), sin(y), cos(y)] """
    def __init__(self, H=64, W=64):
        super().__init__()
        yy, xx = torch.meshgrid(
            torch.linspace(0, 1, H), torch.linspace(0, 1, W), indexing='ij'
        )
        pe = torch.stack([
            torch.sin(2*_m.pi*xx), torch.cos(2*_m.pi*xx),
            torch.sin(2*_m.pi*yy), torch.cos(2*_m.pi*yy)
        ], dim=0)  # [4,H,W]
        self.register_buffer("pe", pe.float(), persistent=False)

    def forward(self, B):
        return self.pe.unsqueeze(0).repeat(B, 1, 1, 1)  # [B,4,H,W]

class ResInceptionDilated(nn.Module):
    """
    Inception 风格多分支 + 膨胀卷积的残差块（无 attention）
    分支：dilation=1,2,4，最后 1x1 fuse 回主通道
    """
    def __init__(self, ch, mid=None):
        super().__init__()
        mid = mid or ch // 2

        self.pre = nn.Sequential(
            nn.GroupNorm(_g(8, ch), ch),
            nn.SiLU()
        )
        self.reduce = nn.Conv2d(ch, mid, 1)

        self.b1 = nn.Conv2d(mid, mid, 3, padding=1, dilation=1, groups=1, bias=False)
        self.b2 = nn.Conv2d(mid, mid, 3, padding=2, dilation=2, groups=1, bias=False)
        self.b3 = nn.Conv2d(mid, mid, 3, padding=4, dilation=4, groups=1, bias=False)

        self.fuse = nn.Sequential(
            nn.GroupNorm(_g(8, mid*3), mid*3),
            nn.SiLU(),
            nn.Conv2d(mid*3, ch, 1)
        )

    def forward(self, x):
        h = self.pre(x)
        h = self.reduce(h)
        h1 = self.b1(h)
        h2 = self.b2(h)
        h3 = self.b3(h)
        h  = torch.cat([h1, h2, h3], dim=1)
        h  = self.fuse(h)
        return x + h

class UpStage(nn.Module):
    """ 双线性上采样 + 3x3 卷积 + 残差精炼 """
    def __init__(self, ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)
        self.rb   = ResInceptionDilated(ch, mid=ch//2)
    def forward(self, x):
        x = self.up(x)
        x = self.conv(x)
        x = self.rb(x)
        return x

class LatentAdapter(nn.Module):
    """
    输入:  z64        [B,4,64,64] (VAE latent * scaling_factor)
          mask64     [B,1,64,64] (可选, 由 bbox 下采样而来; None 表示不使用)
    输出:  dict: {"s64","s128","s256","s512"} 每个 [B,cond_ch,⋅,⋅]
    设计:  64×64 多分支膨胀残差 -> (可选) mask 门控 -> 逐级上采样 + 残差 -> 1×1 投影到 cond_ch
    """
    def __init__(self, cz=4, cond_ch=64, width=128,
                 num_blocks_64=3, include_posenc=True):
        super().__init__()
        self.include_posenc = include_posenc

        in_ch = cz + 4

        # 64×64 输入映射
        self.in_conv = nn.Sequential(
            nn.Conv2d(in_ch, width, 3, padding=1),
            nn.GroupNorm(_g(8, width), width),
            nn.SiLU()
        )
        # 64×64 残差堆叠（多分支膨胀）
        self.blocks64 = nn.Sequential(*[ResInceptionDilated(width, mid=width//2) for _ in range(num_blocks_64)])

        # 金字塔上采样阶段
        self.up1 = UpStage(width)   # -> 128
        self.up2 = UpStage(width)   # -> 256
        self.up3 = UpStage(width)   # -> 512

        # 每级输出到 cond_ch (+ 轻量归一/激活稳定)
        def head():
            return nn.Sequential(
                nn.Conv2d(width, cond_ch, 1),
                nn.GroupNorm(_g(8, cond_ch), cond_ch),
                nn.SiLU()
            )
        self.out64  = head()
        self.out128 = head()
        self.out256 = head()
        self.out512 = head()

        self.posenc = SinCosPos2D(64, 64) if include_posenc else None

    def forward(self, z64):
        """
        z64:    [B,4,64,64]
        mask64: [B,1,64,64] or None
        """
        B, _, H, W = z64.shape
        feats = [z64]


        if self.include_posenc:
            feats.append(self.posenc(B).to(z64.dtype).to(z64.device))

        x = torch.cat(feats, dim=1)             # [B,in_ch,64,64]
        x = self.in_conv(x)                     # [B,width,64,64]
        x = self.blocks64(x)                    # 64×64 膨胀残差

        # 多尺度输出
        f64  = self.out64(x)                    # [B,cond_ch,64,64]
        x128 = self.up1(x)
        f128 = self.out128(x128)                # [B,cond_ch,128,128]
        x256 = self.up2(x128)
        f256 = self.out256(x256)                # [B,cond_ch,256,256]
        x512 = self.up3(x256)
        f512 = self.out512(x512)                # [B,cond_ch,512,512]

        return {"s64": f64, "s128": f128, "s256": f256, "s512": f512}


In [ ]:
def latent_mask_from_bbox(bbox, H=64, W=64):
    """
    bbox: [B,4] with (x1,y1,x2,y2) in [0,1]
    return: [B,1,H,W] with 1 on masked region (to be denoised), 0 elsewhere (keep known)
    """
    B = bbox.shape[0]
    device = bbox.device
    mask = torch.zeros(B, 1, H, W, device=device, dtype=torch.float32)
    for i in range(B):
      x1, y1, x2, y2 = bbox[i]
      x1i, x2i = int(x1.item() * W), int(x2.item() * W)
      y1i, y2i = int(y1.item() * H), int(y2.item() * H)
      mask[i, :, y1i:y2i, x1i:x2i] = 1.0
    return mask

In [ ]:
# ===== 2) 预计算数据的 Dataset（训练 512 级时直接读取，不再跑小级）=====
class PrecomputedCascadeDataset(Dataset):
    def __init__(self, index_jsonl):
        self.items = []
        with open(index_jsonl, "r") as f:
            for line in f:
                path = json.loads(line)["pt"]
                if os.path.exists(path):
                    self.items.append(path)
                else:
                    print(f"[WARN] Missing file, skipped: {path}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        pack = torch.load(self.items[i], map_location="cpu")
        return (
            pack["masked_img"].float(),     # [-1,1]
            pack["target_img"].float(),     # [-1,1]
            torch.tensor(pack["bbox"], dtype=torch.float32),
            pack["z_cond"].to(torch.float16)  # [4,64,64]
        )

# -----------------------------
# UNet512：像素域 512×512，条件拼接（不改损失）
# -----------------------------
def timestep_embedding(timesteps, dim, max_period=10000):
    # diffusers风格的正余弦时间步嵌入
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(0, half, dtype=torch.float32, device=timesteps.device) / half)
    args = timesteps.float()[:, None] * freqs[None]
    emb = torch.cat([torch.cos(args), torch.sin(args)], dim=1)
    if dim % 2:
        emb = torch.cat([emb, emb[:, :1]], dim=1)
    return emb

class Down(nn.Module):
    def __init__(self, in_ch, out_ch, tdim, cond_ch=0):
        super().__init__()
        self.block1 = ResidualBlock2d(in_ch + cond_ch, out_ch, time_dim=tdim)
        self.block2 = ResidualBlock2d(out_ch, out_ch, time_dim=tdim)
        self.down = nn.AvgPool2d(2)
    def forward(self, x, t_emb, cond=None):
        if cond is not None:
            x = torch.cat([x, cond], dim=1)
        x = self.block1(x, t_emb)
        x = self.block2(x, t_emb)
        x_down = self.down(x)
        return x, x_down  # return skip, next

class Up(nn.Module):
    def __init__(self, in_ch, out_ch, tdim, cond_ch=0):
        super().__init__()
        self.block1 = ResidualBlock2d(in_ch + cond_ch, out_ch, time_dim=tdim)
        self.block2 = ResidualBlock2d(out_ch, out_ch, time_dim=tdim)
    def forward(self, x, skip, t_emb, cond=None):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        if cond is not None:
            x = torch.cat([x, cond], dim=1)
        x = self.block1(x, t_emb)
        x = self.block2(x, t_emb)
        return x

from timm.models.vision_transformer import Attention as ViTAttention
import torch.nn as nn
import math as _m

def _g(n, c):  # GroupNorm自适应分组
    return _m.gcd(n, c) or 1

class TimmAttn2D(nn.Module):
    """把 timm 的 ViT Attention 用在 2D 特征上（不重写注意力本体）"""
    def __init__(self, dim, num_heads=4, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.norm = nn.GroupNorm(_g(8, dim), dim)
        self.attn = ViTAttention(dim=dim, num_heads=num_heads,
                                 qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=proj_drop)

    def forward(self, x):  # x: [B,C,H,W]
        B, C, H, W = x.shape
        x = self.norm(x)
        x = x.flatten(2).transpose(1, 2)   # [B, HW, C]
        x = self.attn(x)                   # [B, HW, C] （来自 timm）
        x = x.transpose(1, 2).reshape(B, C, H, W)
        return x


class UNet512(nn.Module):
    """
    输入: x_noisy [B,3,512,512], t
    条件: cond_feats dict {s64,s128,s256,s512}，每个 [B,cond_ch,scale,scale]
    最小下采样到 64×64（无 32×32）
    """
    def __init__(self, base_ch=128, cond_ch=64, time_dim=256):
        super().__init__()
        ch1, ch2, ch3 = base_ch, base_ch*2, base_ch*4  # 不再用 ch4

        self.time_mlp = nn.Sequential(
            nn.Linear(320, time_dim), nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        # 输入头：拼 s512 条件
        self.in_conv = nn.Conv2d(3 + cond_ch, ch1, 3, padding=1)

        # 下采样路径：512->256->128->64
        self.down1 = Down(ch1, ch1, tdim=time_dim, cond_ch=0)           # 512->256
        self.down2 = Down(ch1, ch2, tdim=time_dim, cond_ch=cond_ch)     # 256->128
        self.down3 = Down(ch2, ch3, tdim=time_dim, cond_ch=cond_ch)     # 128->64

        # 64×64 瓶颈注意力（可保留/去掉，按你之前的实现）
        self.attn64 = TimmAttn2D(dim=ch3, num_heads=4)

        # 64×64 瓶颈残差：在 64 处与 s64 条件融合
        self.mid1 = ResidualBlock2d(ch3 + cond_ch, ch3, time_dim=time_dim)
        self.mid2 = ResidualBlock2d(ch3, ch3, time_dim=time_dim)

        # 上采样路径：64->128->256->512
        self.up3 = Up(ch3 + ch3, ch2, tdim=time_dim, cond_ch=cond_ch)   # 64->128（与 skip3 对齐）
        self.up2 = Up(ch2 + ch2, ch1, tdim=time_dim, cond_ch=cond_ch)   # 128->256（与 skip2 对齐）
        self.up1 = Up(ch1 + ch1, ch1, tdim=time_dim, cond_ch=0)         # 256->512（与 skip1 对齐）

        self.out_norm = nn.GroupNorm(8, ch1)
        self.out = nn.Conv2d(ch1, 3, 3, padding=1)

    def forward(self, x, timesteps, cond_feats):
        # t embedding（timesteps 需为 [B] 的 LongTensor）
        t_emb = self.time_mlp(timestep_embedding(timesteps, 320))

        # 输入拼接 s512 条件
        x = torch.cat([x, cond_feats["s512"]], dim=1)
        x = self.in_conv(x)

        # Down: 512->256->128->64（逐层注入 s256 / s128）
        skip1, x = self.down1(x, t_emb, cond=None)                       # 512->256
        skip2, x = self.down2(x, t_emb, cond=cond_feats["s256"])         # 256->128
        skip3, x = self.down3(x, t_emb, cond=cond_feats["s128"])         # 128->64

        # 64×64 瓶颈
        x = self.attn64(x)                                               # ★ 64×64 注意力
        x = torch.cat([x, cond_feats["s64"]], dim=1)                     # 与 s64 条件融合
        x = self.mid1(x, t_emb)
        x = self.mid2(x, t_emb)

        # Up: 64->128->256->512（逐层注入 s128 / s256）
        x = self.up3(x, skip3, t_emb, cond=cond_feats["s128"])           # 64->128
        x = self.up2(x, skip2, t_emb, cond=cond_feats["s256"])           # 128->256
        x = self.up1(x, skip1, t_emb, cond=None)                         # 256->512

        x = self.out(self.out_norm(x).clamp(-6, 6))
        x = torch.tanh(x)
        return x  # 预测 x0

# -----------------------------
# 级联训练器（只训练大模型）
# -----------------------------
class Cascade512Trainer:
    def __init__(self, train_index, val_index,bs=4, lr=1e-5):
        self.accelerator = Accelerator(mixed_precision='fp16')
        self.loss_history, self.val_loss_history = [], []

        self.train_loader = DataLoader(PrecomputedCascadeDataset(train_index), batch_size=bs, shuffle=True, num_workers=4, pin_memory=True)
        self.val_loader   = DataLoader(PrecomputedCascadeDataset(val_index),   batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

        self.adapter = LatentAdapter(cz=4, cond_ch=64)
        self.unet512 = UNet512(base_ch=128, cond_ch=64, time_dim=256)

        self.optimizer = torch.optim.AdamW(list(self.adapter.parameters())+list(self.unet512.parameters()),
                                           lr=lr, betas=(0.9,0.999), weight_decay=1e-5)
        self.scheduler2 = DDPMScheduler.from_pretrained("runwayml/stable-diffusion-v1-5", subfolder="scheduler")
        self.scheduler2.config.prediction_type = "sample"

        comps = [self.adapter, self.unet512, self.train_loader, self.val_loader, self.optimizer]
        self.adapter, self.unet512, self.train_loader, self.val_loader, self.optimizer = self.accelerator.prepare(*comps)

        self.vis_dir = "drive/MyDrive/vis_cascade"
        os.makedirs(self.vis_dir, exist_ok=True)

    def _pixel_mask_from_bbox(self, bbox, img_shape):
        B, _, H, W = img_shape
        masks = torch.zeros(B, 1, H, W, device=self.accelerator.device)
        for i in range(B):
            x1,y1,x2,y2 = bbox[i]
            x1i,x2i = int(x1.item()*W), int(x2.item()*W)
            y1i,y2i = int(y1.item()*H), int(y2.item()*H)
            masks[i, :, y1i:y2i, x1i:x2i] = 1.0
        return masks

    def _step(self, batch, train=True):
        masked_imgs, target_imgs, bbox, z_cond = batch
        #z_cond: [B,4,64,64] (float16 on CPU) → to device/float16
        z_cond = z_cond.to(self.accelerator.device, dtype=torch.float16)
        cond_feats = self.adapter(z_cond)
        noise = torch.randn_like(target_imgs)
        timesteps = torch.randint(0, self.scheduler2.config.num_train_timesteps,
                                  (target_imgs.shape[0],), device=target_imgs.device).long()
        x_noisy = self.scheduler2.add_noise(target_imgs, noise, timesteps)
        pixel_mask = self._pixel_mask_from_bbox(bbox, target_imgs.shape)
        x0_pred = self.unet512(x_noisy, timesteps, cond_feats)
        loss = F.mse_loss(x0_pred, target_imgs)# + 0.2 * F.l1_loss(x0_pred, target_imgs)

        if train:
            self.accelerator.backward(loss)
        return loss

    @torch.no_grad()
    def validate(self):
        self.unet512.eval(); self.adapter.eval()
        total, n = 0.0, 0
        for batch in tqdm(self.val_loader, desc="Validating"):
            loss = self._step(batch, train=False)
            total += loss.item(); n += 1
        return total / max(1,n)

    @torch.no_grad()
    def visualize_epoch(self, epoch_idx:int, max_batches:int=1, steps_stage2:int=50):
        self.unet512.eval(); self.adapter.eval()
        saved = 0; grids=[]
        self.scheduler2.set_timesteps(steps_stage2, device=self.accelerator.device)
        for batch in self.val_loader:
            masked_imgs, target_imgs, bbox, z_cond = batch
            z_cond = z_cond.to(self.accelerator.device, dtype=torch.float16)
            cond_feats = self.adapter(z_cond)

            B,_,H,W = masked_imgs.shape
            x = torch.randn(B,3,H,W, device=self.accelerator.device) * self.scheduler2.init_noise_sigma
            for t in self.scheduler2.timesteps:
                t_batch = torch.full((B,), int(t), device=self.accelerator.device, dtype=torch.long)
                x0_pred = self.unet512(x, t_batch, cond_feats)
                x = self.scheduler2.step(x0_pred, t, x).prev_sample

            pred = (x.clamp(-1,1)+1)/2
            masked_vis = (masked_imgs.clamp(-1,1)+1)/2
            target_vis = (target_imgs.clamp(-1,1)+1)/2
            triplet = torch.cat([masked_vis, target_vis, pred], dim=0)
            grid = vutils.make_grid(triplet, nrow=B, padding=2)
            grids.append(grid); saved += 1
            if saved>=max_batches: break
        if grids:
            big = torch.cat(grids, dim=1) if len(grids)>1 else grids[0]
            out_path = os.path.join(self.vis_dir, f"epoch_{epoch_idx:03d}.png")
            vutils.save_image(big, out_path)
            self.accelerator.print(f"[Visualize] Saved {out_path}")

    @torch.no_grad()
    def eval_composition_batch(self, ae_model, index, steps_stage2=50, save_dir="comp_eval"):
        self.unet512.eval(); self.adapter.eval()
        os.makedirs(save_dir, exist_ok=True)

        self.scheduler2.set_timesteps(steps_stage2, device=self.accelerator.device)

        # 从验证集取一个 batch
        batch = next(iter(self.val_loader))
        masked_imgs, target_imgs, bbox, z_cond = batch
        B = masked_imgs.size(0)

        z_cond = z_cond.to(self.accelerator.device, dtype=torch.float16)
        cond_feats = self.adapter(z_cond)

        # 扩散生成
        x = torch.randn(B, 3, masked_imgs.shape[2], masked_imgs.shape[3],
                    device=self.accelerator.device) * self.scheduler2.init_noise_sigma
        for t in self.scheduler2.timesteps:
            t_batch = torch.full((B,), int(t), device=self.accelerator.device, dtype=torch.long)
            x0_pred = self.unet512(x, t_batch, cond_feats)
            x = self.scheduler2.step(x0_pred, t, x).prev_sample

        pred_imgs = (x.clamp(-1,1)+1)/2
        masked_vis = (masked_imgs.clamp(-1,1)+1)/2
        target_vis = (target_imgs.clamp(-1,1)+1)/2

        # 计算 composition 分布（使用 infer_cell_map）
        fr_recon = []
        fr_pred = []
        fr_orig = []

        for i in range(B):
            type_recon = infer_cell_map(masked_vis[i], ae_model)
            type_pred  = infer_cell_map(pred_imgs[i], ae_model)
            type_orig  = infer_cell_map(target_vis[i], ae_model)

            dist_recon = compute_type_distribution(type_recon.squeeze(0).cpu().numpy(), num_types=25)
            dist_pred  = compute_type_distribution(type_pred.squeeze(0).cpu().numpy(), num_types=25)
            dist_orig  = compute_type_distribution(type_orig.squeeze(0).cpu().numpy(), num_types=25)

            fr_recon.append(dist_recon)
            fr_pred.append(dist_pred)
            fr_orig.append(dist_orig)

        # 保存可视化图
        for i in range(B):
            # 在原图三图和 composition 三图之后，增加 RGB 分布三图

            # 创建 3x3 子图（原有的 2x3 → 改成 3x3）
            fig, axes = plt.subplots(3, 3, figsize=(12, 8))
            def show_img(ax, img, title):
              ax.imshow(img.permute(1, 2, 0).cpu().numpy())
              ax.set_title(title)
              ax.axis("off")

            # ===== 第一行：图像可视化 =====
            show_img(axes[0,0], masked_vis[i], "Recon (cond)")
            show_img(axes[0,1], target_vis[i], "Orig (GT)")
            show_img(axes[0,2], pred_imgs[i], "Pred")

            # ===== 第二行：composition 柱状图 =====
            xs = np.arange(25)
            bar_width = 0.8

            for ax, frac, title in zip(axes[1],
                [fr_recon[i], fr_orig[i], fr_pred[i]],
                ["Recon comp", "Orig comp", "Pred comp"]):
                ax.bar(xs, frac, width=bar_width, color='skyblue')
                ax.set_title(title)
                ax.set_ylim(0, 1)
                ax.set_xticks(xs)
                ax.set_xticklabels([f"T{i+1}" for i in xs], rotation=90, fontsize=6)
                ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
                ax.set_yticklabels(["0%", "20%", "40%", "60%", "80%", "100%"])
                ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.3)

            # ===== 第三行：RGB 直方图 =====
            for ax, img, title in zip(axes[2],
                    [masked_vis[i], target_vis[i], pred_imgs[i]],
                    ["Recon RGB Hist", "Orig RGB Hist", "Pred RGB Hist"]):

                img_np = img.cpu().numpy()  # [3,H,W]
                colors = ['r', 'g', 'b']
                for c in range(3):
                    hist, bins = np.histogram(img_np[c], bins=50, range=(0,1))
                    ax.plot(bins[:-1], hist, color=colors[c], alpha=0.7, label=colors[c].upper())
                ax.set_title(title)
                ax.set_xlim(0, 1)
                ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
                ax.set_yticks([])
                ax.legend()

            plt.tight_layout()
            out_path = os.path.join(save_dir, f"vis_comp_img{i}_{index}.png")
            plt.savefig(out_path, dpi=150)
            plt.close()
            self.accelerator.print(f"[CompEval] Saved {out_path}")




    def train(self, epochs=30, patience=5, vis_steps_stage2=50):
        device = self.accelerator.device
        ae = Autoencoder().to(device).eval()
        ae.load_state_dict(torch.load("drive/MyDrive/newae2.pth", map_location=device))
        best = float("inf"); bad=0
        for ep in range(1, epochs+1):
            self.unet512.train(); self.adapter.train()
            prog = tqdm(self.train_loader, desc=f"Epoch {ep} [Train]")
            losses=[]
            for batch in prog:
                with self.accelerator.accumulate(self.unet512):
                    loss = self._step(batch, train=True)
                    if self.accelerator.sync_gradients:
                        self.accelerator.clip_grad_norm_(self.unet512.parameters(), 1.0)
                    self.optimizer.step(); self.optimizer.zero_grad()
                losses.append(loss.item()); prog.set_postfix(loss=np.mean(losses))

            tr = float(np.mean(losses)); self.loss_history.append(tr)
            va = self.validate(); self.val_loss_history.append(va)
            self.accelerator.print(f"[Epoch {ep}] train={tr:.4f} val={va:.4f}")
            self.visualize_epoch(epoch_idx=ep, max_batches=4, steps_stage2=vis_steps_stage2)
            self.eval_composition_batch(ae, ep, steps_stage2=vis_steps_stage2, save_dir = "drive/MyDrive/checkpoint_cascade512_best")

            if va < best - 1e-4:
                best = va; bad=0
                self.accelerator.wait_for_everyone()
                self.accelerator.save_state("drive/MyDrive/checkpoint_cascade512_best")
                self.accelerator.print(f"  >> Saved best at val {best:.4f}")
            else:
                bad += 1; self.accelerator.print(f"  >> No improve ({bad}/{patience})")
                if bad>=patience:
                    self.accelerator.print("Early stop."); break


In [ ]:
train_index = "drive/MyDrive/precomputed_cascade/train/index.jsonl"
val_index   = "drive/MyDrive/precomputed_cascade/val/index.jsonl"

trainer = Cascade512Trainer(train_index, val_index, bs=4, lr=2e-5)
#trainer.accelerator.load_state("drive/MyDrive/checkpoint_cascade512_best")


[WARN] Missing file, skipped: drive/MyDrive/precomputed_cascade/train/region_20_k000.pt
[WARN] Missing file, skipped: drive/MyDrive/precomputed_cascade/train/region_20_k001.pt
[WARN] Missing file, skipped: drive/MyDrive/precomputed_cascade/train/region_20_k002.pt
[WARN] Missing file, skipped: drive/MyDrive/precomputed_cascade/train/region_20_k003.pt
[WARN] Missing file, skipped: drive/MyDrive/precomputed_cascade/train/region_20_k004.pt
[WARN] Missing file, skipped: drive/MyDrive/precomputed_cascade/train/region_20_k005.pt
[WARN] Missing file, skipped: drive/MyDrive/precomputed_cascade/train/region_20_k006.pt
[WARN] Missing file, skipped: drive/MyDrive/precomputed_cascade/train/region_20_k007.pt
[WARN] Missing file, skipped: drive/MyDrive/precomputed_cascade/train/region_20_k008.pt
[WARN] Missing file, skipped: drive/MyDrive/precomputed_cascade/train/region_20_k009.pt
[WARN] Missing file, skipped: drive/MyDrive/precomputed_cascade/train/region_20_k010.pt
[WARN] Missing file, skipped: dr

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

In [ ]:
trainer.train(epochs=30, patience=10, vis_steps_stage2=200)

Epoch 1 [Train]:   0%|          | 0/940 [00:00<?, ?it/s]

Validating:   0%|          | 0/80 [00:00<?, ?it/s]

[Epoch 1] train=0.0898 val=0.1335
[Visualize] Saved drive/MyDrive/vis_cascade/epoch_001.png
原始图像 RGB min/max：
  R: 0.01171875 1.0
  G: 0.00390625 1.0
  B: 0.0234375 1.0
有效像素数量（非白）： 95460
原始图像 RGB min/max：
  R: 0.017578125 1.0
  G: 0.00439453125 1.0
  B: 0.005615234375 1.0
有效像素数量（非白）： 107699
原始图像 RGB min/max：
  R: 0.01171875 1.0
  G: 0.00390625 1.0
  B: 0.0234375 1.0
有效像素数量（非白）： 76044
原始图像 RGB min/max：
  R: 0.01171875 1.0
  G: 0.00390625 1.0
  B: 0.0234375 1.0
有效像素数量（非白）： 77854
原始图像 RGB min/max：
  R: 0.026123046875 1.0
  G: 0.00439453125 1.0
  B: 0.005126953125 1.0
有效像素数量（非白）： 105977
原始图像 RGB min/max：
  R: 0.01171875 1.0
  G: 0.00390625 1.0
  B: 0.0234375 1.0
有效像素数量（非白）： 76044
[CompEval] Saved drive/MyDrive/checkpoint_cascade512_best/vis_comp_img0_1.png
[CompEval] Saved drive/MyDrive/checkpoint_cascade512_best/vis_comp_img1_1.png
  >> Saved best at val 0.1335


Epoch 2 [Train]:   0%|          | 0/940 [00:00<?, ?it/s]

Validating:   0%|          | 0/80 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/80 [00:00<?, ?it/s]

[Epoch 12] train=0.0667 val=0.1184
[Visualize] Saved drive/MyDrive/vis_cascade/epoch_012.png
原始图像 RGB min/max：
  R: 0.01171875 1.0
  G: 0.00390625 1.0
  B: 0.0234375 1.0
有效像素数量（非白）： 95460
原始图像 RGB min/max：
  R: 0.035400390625 1.0
  G: 0.082763671875 1.0
  B: 0.08544921875 1.0
有效像素数量（非白）： 74791
原始图像 RGB min/max：
  R: 0.01171875 1.0
  G: 0.00390625 1.0
  B: 0.0234375 1.0
有效像素数量（非白）： 76044
原始图像 RGB min/max：
  R: 0.01171875 1.0
  G: 0.00390625 1.0
  B: 0.0234375 1.0
有效像素数量（非白）： 77854
原始图像 RGB min/max：
  R: 0.03076171875 1.0
  G: 0.089599609375 1.0
  B: 0.08544921875 1.0
有效像素数量（非白）： 73222
原始图像 RGB min/max：
  R: 0.01171875 1.0
  G: 0.00390625 1.0
  B: 0.0234375 1.0
有效像素数量（非白）： 76044
[CompEval] Saved drive/MyDrive/checkpoint_cascade512_best/vis_comp_img0_12.png
[CompEval] Saved drive/MyDrive/checkpoint_cascade512_best/vis_comp_img1_12.png
  >> No improve (9/10)


Epoch 13 [Train]:   0%|          | 0/940 [00:00<?, ?it/s]

Validating:   0%|          | 0/80 [00:00<?, ?it/s]

[Epoch 13] train=0.0654 val=0.1112
[Visualize] Saved drive/MyDrive/vis_cascade/epoch_013.png
原始图像 RGB min/max：
  R: 0.01171875 1.0
  G: 0.00390625 1.0
  B: 0.0234375 1.0
有效像素数量（非白）： 95460
原始图像 RGB min/max：
  R: 0.02490234375 1.0
  G: 0.09765625 1.0
  B: 0.07373046875 1.0
有效像素数量（非白）： 69044
原始图像 RGB min/max：
  R: 0.01171875 1.0
  G: 0.00390625 1.0
  B: 0.0234375 1.0
有效像素数量（非白）： 76044
原始图像 RGB min/max：
  R: 0.01171875 1.0
  G: 0.00390625 1.0
  B: 0.0234375 1.0
有效像素数量（非白）： 77854
原始图像 RGB min/max：
  R: 0.01513671875 1.0
  G: 0.089599609375 1.0
  B: 0.07763671875 1.0
有效像素数量（非白）： 68035
原始图像 RGB min/max：
  R: 0.01171875 1.0
  G: 0.00390625 1.0
  B: 0.0234375 1.0
有效像素数量（非白）： 76044
[CompEval] Saved drive/MyDrive/checkpoint_cascade512_best/vis_comp_img0_13.png
[CompEval] Saved drive/MyDrive/checkpoint_cascade512_best/vis_comp_img1_13.png
  >> No improve (10/10)
Early stop.


In [ ]:
trainer.accelerator.wait_for_everyone()
trainer.accelerator.save_state("drive/MyDrive/checkpoint_cascade512_best")

PosixPath('drive/MyDrive/checkpoint_cascade512_best')

In [ ]:
ae = Autoencoder().to("cuda").eval()
ae.load_state_dict(torch.load("drive/MyDrive/newae2.pth", map_location="cuda"))
x = trainer.eval_composition_batch(ae, 200)

In [ ]:
def load_and_recover_z3d_png(path):
    """
    Load a PNG image as RGB, interpret it as normalized z_3d embedding encoded as uint8 [0–255],
    and recover the original z_3d float values via inverse scaling.
    """
    # 1. Load RGB image as uint8 tensor
    img = Image.open(path).convert("RGB")
    arr = torch.ByteTensor(torch.ByteStorage.from_buffer(img.tobytes()))
    rgb = arr.view(img.size[1], img.size[0], 3).permute(2, 0, 1).clone()  # [3, H, W], uint8
    mask = (rgb > 230).all(dim=0)
    rgb[:, mask] = 255
    rgb_float = rgb.float() / 255.0  # [3, H, W], float32
    non_white_pixels = ((rgb_float != 1).any(dim=0)).sum().item()
    print(f"count for non white: {non_white_pixels}")

    return rgb_float


In [ ]:
def infer_cell_map(latent_image, model):
    min_vals = torch.tensor([-69.761505, -75.65188,  -77.16103], device=latent_image.device)
    range_vals = torch.tensor([88.969406, 65.244896, 67.13518], device=latent_image.device) - min_vals
    H, W = latent_image.shape[1], latent_image.shape[2]
    latent_image = latent_image.to("cuda")
    range_vals = range_vals.to("cuda")
    min_vals = min_vals.to("cuda")

    print("原始图像 RGB min/max：")
    print("  R:", latent_image[0].min().item(), latent_image[0].max().item())
    print("  G:", latent_image[1].min().item(), latent_image[1].max().item())
    print("  B:", latent_image[2].min().item(), latent_image[2].max().item())

    flat_img = latent_image.permute(1, 2, 0).reshape(-1, 3)
    white_mask = (flat_img > 0.95).all(dim=1)

    print("有效像素数量（非白）：", (~white_mask).sum().item())

    infer_input_rgb = flat_img[~white_mask]
    pred = torch.zeros(flat_img.shape[0], dtype=torch.long, device="cuda")
    if infer_input_rgb.shape[0] > 0:
        z_recovered = infer_input_rgb * range_vals + min_vals
        logits = model.decoder(z_recovered)
        pred[~white_mask] = torch.argmax(logits, dim=1) + 1

    pred = pred.reshape(1, H, W)
    return pred


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from collections import defaultdict

# -------------------- 基本配置 -------------------- #
device = "cuda"

cell_type_names = [
    "B", "CD4+ T cell", "CD57+ Enterocyte", "CD66+ Enterocyte", "CD7+ Immune",
    "CD8+ T", "Cycling TA", "DC", "Endothelial", "Enterocyte",
    "Goblet", "ICC", "Lymphatic", "M1 Macrophage", "M2 Macrophage",
    "MUC1+ Enterocyte", "NK", "Nerve", "Neuroendocrine", "Neutrophil",
    "Paneth", "Plasma", "Smooth muscle", "Stroma", "TA"
]  # 索引 1..25

type_color_dict = {
    1: [134.9, 124.1, 8.7],
    2: [62.3, 216.0, 132.0],
    3: [130.9, 46.4, 182.5],
    4: [92.8, 63.6, 202.5],
    5: [140.1, 239.7, 189.0],
    6: [200.0, 183.1, 168.9],
    7: [108.3, 16.1, 142.3],
    8: [132.7, 217.1, 223.4],
    9: [170.6, 96.0, 98.0],
    10: [68.4, 101.1, 101.3],
    11: [4.6, 146.9, 145.6],
    12: [177.1, 1.5, 112.2],
    13: [202.6, 136.4, 12.3],
    14: [95.1, 244.3, 209.6],
    15: [32.5, 214.2, 217.8],
    16: [176.2, 78.6, 193.2],
    17: [185.9, 206.0, 77.8],
    18: [212.0, 39.3, 35.0],
    19: [117.1, 190.4, 53.4],
    20: [101.2, 145.2, 246.9],
    21: [13.7, 179.1, 133.2],
    22: [247.4, 125.5, 111.5],
    23: [122.9, 151.0, 150.0],
    24: [118.0, 45.6, 47.7],
    25: [31.1, 114.7, 222.5]
}

# 颜色从 0-255 转 [0,1]
type_color_norm = {
    idx: (np.array(rgb) / 255.0).tolist() for idx, rgb in type_color_dict.items()
}

# -------------------- 工具函数 -------------------- #
def compute_type_distribution(type_map_np: np.ndarray, num_types: int = 25) -> np.ndarray:
    """
    计算 1..num_types 的类别比例，自动忽略 0（背景）。
    返回长度=num_types 的比例向量，缺失类别为 0。
    """
    counts = np.bincount(type_map_np.ravel(), minlength=num_types + 1)[1:num_types + 1].astype(float)
    total = counts.sum()
    if total <= 0:
        return np.zeros(num_types, dtype=float)
    return counts / total

def save_inferred_map_to_df(cell_type_map: torch.Tensor, region_id=0) -> pd.DataFrame:
    """
    将推理得到的 cell type map（[1,H,W]或[H,W]）转为 DataFrame。
    仅保留非背景（>0），把 1..25 映射为具体名字。
    """
    if cell_type_map.ndim == 3:
        cell_type_map = cell_type_map.squeeze(0)  # [H, W]
    cell_type_np = cell_type_map.detach().cpu().numpy()  # [H, W]
    ys, xs = np.where(cell_type_np > 0)
    types = cell_type_np[ys, xs].astype(int)
    df = pd.DataFrame({
        "Cell Type": [cell_type_names[t - 1] for t in types],
        "x": xs,
        "y": ys,
        "unique_region": region_id
    })
    return df
